# Lesson 10: capstone

**Goal:** design, train, evaluate and ship a harder text → JSON extractor. You design the training data. The acceptance test is fixed.

**Acceptance bars (on the 60-row acceptance test, greedy decoding)**
- valid-JSON rate **≥ 0.95**
- field accuracy **≥ 0.90** over `name`, `age`, `city` and `email` (a missing email must come out as `null`)
- the model and its card are on the Hub, and the last cell loads it by repo id

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU*. Add your `HF_TOKEN` secret as in Lesson 9.
2. Run the cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code. There are four.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.
4. Missed a bar? Read the failures in section 4, change your data in section 2, and rerun from there. Each loop takes a few minutes.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup and the course data
You'll reuse `NAMES`, `CITIES` and `TEMPLATES` from the course data as a starting point.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

In [ ]:
import json, random

PROMPT = "Extract the person's name, age and city as JSON."
NAMES = ["Sara", "Omar", "Lina", "Youssef", "Mona", "Karim", "Nour", "Emma", "Liam", "Sofia",
         "Mateo", "Aiko", "Kenji", "Priya", "Arjun", "Chloe", "Lucas", "Fatima", "Diego", "Elena",
         "Leo", "Amara", "Kofi", "Hana", "Ines"]          # last 5 are test-only
CITIES = ["Cairo", "Alexandria", "Paris", "Lagos", "Tokyo", "Lima", "Berlin", "Toronto",
          "Mumbai", "Seoul", "Nairobi", "Madrid", "Dubai", "Oslo", "Austin", "Lisbon",
          "Hanoi", "Accra", "Quito", "Perth"]                # last 4 are test-only
TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}.",
    "{name} ({age}) lives in {city}.",
    "My name is {name}, I am {age} years old and I live in {city}.",
    "New signup: {name}, age {age}, based in {city}.",
    "Hello from {city}! {name} here, just turned {age}.",
    "{city}-based {name}, {age}, is joining the team next week.",
    "Contact: {name} | {age} | {city}",
    "Applicant {name} is {age} and currently resides in {city}.",
    "Meet {name}: {age} years young and loving life in {city}.",   # test-only
    "I moved to {city} last year. Name's {name}, age {age}.",      # test-only
]

def make_rows(n, split="train", seed=0):
    """Return n chat examples in TRL's prompt-completion format."""
    rng = random.Random(seed)
    test = split == "test"
    names = NAMES[20:] if test else NAMES[:20]
    cities = CITIES[16:] if test else CITIES[:16]
    templates = TEMPLATES[8:] if test else TEMPLATES[:8]
    rows = []
    for _ in range(n):
        person = {"name": rng.choice(names), "age": rng.randint(18, 80), "city": rng.choice(cities)}
        text = rng.choice(templates).format(**person)
        rows.append({
            "prompt": [{"role": "user", "content": f"{PROMPT}\n\nText: {text}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}],
        })
    return rows

## 1 · The acceptance test (don't edit)
The harder schema adds `email`, which is `null` when the message has none. The test uses names, cities, email domains and phrasings that must **not** appear in your training data. Some names and cities have two words, and some messages contain numbers that aren't the age.

`score_v2` is the course `score()` generalised to any set of fields. It also returns the failing rows, for error analysis.

In [ ]:
PROMPT_V2 = "Extract the person's name, age, city and email as JSON. Use null for anything missing."

def fill(template, name, rng, cities, domains):
    """One prompt-completion row. The email is null when the template has no {email} slot."""
    email = f"{name.lower().replace(' ', '.')}{rng.randint(1, 99)}@{rng.choice(domains)}"
    person = {"name": name, "age": rng.randint(18, 80), "city": rng.choice(cities),
              "email": email if "{email}" in template else None}
    return {"prompt": [{"role": "user", "content": f"{PROMPT_V2}\n\nText: {template.format(**person)}"}],
            "completion": [{"role": "assistant", "content": json.dumps(person)}]}

TEST_NAMES = ["Wanjiru Kamau", "Tomas", "Mei Lin", "Rafael Souza", "Zainab", "Olu Adeyemi", "Ingrid", "Kwame Mensah"]
TEST_CITIES = ["Rio de Janeiro", "Kuala Lumpur", "Buenos Aires", "Addis Ababa", "Quito", "Perth", "Hanoi", "Accra"]
TEST_DOMAINS = ["yahoo.com", "fastmail.fm", "uni.edu", "company.org"]
TEST_TEMPLATES = [
    "hey its {name} here, {age}, living in {city} now. reach me at {email}",
    "{name} - {age} yrs - {city} - {email}",
    "Quick intro: {name} ({age}) from {city}. Has 2 dogs and 3 cats.",
    "Forwarding this: {name}, {age}, currently in {city}. No email on file.",
    "Email {email} belongs to {name}, {age}, who lives in {city}.",
    "{name} just turned {age}! Moved to {city} in 2019.",
]

def make_test_v2(n=60, seed=123):
    rng = random.Random(seed)
    return [fill(rng.choice(TEST_TEMPLATES), rng.choice(TEST_NAMES), rng, TEST_CITIES, TEST_DOMAINS)
            for _ in range(n)]

def score_v2(model, tokenizer, rows, show=3, max_new_tokens=60):
    """Like score(), for any fields. Returns (scores, failures)."""
    model.eval()
    valid, right, total, failures = 0, 0, 0, []
    for i, row in enumerate(rows):
        gold = json.loads(row["completion"][0]["content"])
        inputs = tokenizer.apply_chat_template(row["prompt"], add_generation_prompt=True,
                                               return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            pred = json.loads(reply)
            valid += 1
        except json.JSONDecodeError:
            pred = {}
        if not isinstance(pred, dict):
            pred = {}
        ok = sum(pred.get(k) == v for k, v in gold.items())
        right, total = right + ok, total + len(gold)
        if ok < len(gold):
            failures.append({"text": row["prompt"][0]["content"].split("Text: ", 1)[1], "reply": reply,
                             "expected": row["completion"][0]["content"]})
        if i < show:
            print(f"  reply: {reply!r}")
    scores = {"valid_json": valid / len(rows), "field_accuracy": right / total}
    print(scores)
    return scores, failures

test = make_test_v2()
for row in test[:3]:
    print(row["prompt"][0]["content"].split("Text: ", 1)[1], "\n  ->", row["completion"][0]["content"])

## 2 · Design your training data
This is the capstone. Your Lesson 8 failure tags are your checklist:
- `invented` → some messages must leave the email out, so the model learns `null`.
- `wrong_value` → show it the patterns it will meet: two-word names and cities, numbers that aren't the age, messy casing.
- Never copy test names, cities, domains or phrasings. The check below enforces this.

**TODO 1:** Write at least 8 training phrasings in `TRAIN_TEMPLATES`. Every template needs `{name}`, `{age}` and `{city}`. Put `{email}` in some and leave it out of others. Extend the name, city and domain lists too.

In [ ]:
TRAIN_NAMES = NAMES[:20]
TRAIN_CITIES = CITIES[:16]
TRAIN_DOMAINS = ["gmail.com", "outlook.com", "proton.me"]

# TODO 1: at least 8 phrasings, one per line
TRAIN_TEMPLATES = [
    "Hi, I'm {name} from {city}. I'm {age}. Email: {email}",
    "{name} ({age}) lives in {city}.",
    ...,
]

def make_train_v2(n, seed=0):
    rng = random.Random(seed)
    templates = [t for t in TRAIN_TEMPLATES if t is not ...]
    return [fill(rng.choice(templates), rng.choice(TRAIN_NAMES), rng, TRAIN_CITIES, TRAIN_DOMAINS)
            for _ in range(n)]

for row in make_train_v2(5, seed=1):
    print(row["prompt"][0]["content"].split("Text: ", 1)[1], "\n  ->", row["completion"][0]["content"])

In [ ]:
# check 1
templates = [t for t in TRAIN_TEMPLATES if t is not ...]
complete = all(isinstance(t, str) and all(s in t for s in ("{name}", "{age}", "{city}")) for t in templates)
with_email = sum("{email}" in t for t in templates)
check(len(templates) >= 8 and complete, f"{len(templates)} phrasings, each with name, age and city.",
      "Write at least 8 template strings. Each needs {name}, {age} and {city}.")
check(with_email >= 2 and len(templates) - with_email >= 2, "Some phrasings have an email and some don't.",
      "The model only learns null if it sees messages without an email. Mix both kinds.")
leaks = (set(TRAIN_NAMES) & set(TEST_NAMES)) | (set(TRAIN_CITIES) & set(TEST_CITIES)) \
        | (set(TRAIN_DOMAINS) & set(TEST_DOMAINS)) | (set(templates) & set(TEST_TEMPLATES))
check(not leaks, "Nothing from the acceptance test leaked into training.",
      f"Remove these from your training lists: {sorted(leaks)}")

## 3 · Train
Pick a recipe: Lesson 5 (full SFT of the 135M model) or Lesson 6 (LoRA on the 360M model). Start small and fast. Go bigger only if the bars demand it.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"    # or "HuggingFaceTB/SmolLM2-360M-Instruct" with LoRA
N_TRAIN = 1000

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)
train = Dataset.from_list(make_train_v2(N_TRAIN))

print("BASE model on the acceptance test:")
before, _ = score_v2(model, tok, test, show=1)

**TODO 2:** Build `args` and `trainer` from your chosen lesson. For LoRA, also pass a `peft_config` (Lesson 6). Keep `bf16=False` for the T4.

In [ ]:
# TODO 2
args = ...
trainer = ...

In [ ]:
# check 2
check(isinstance(trainer, SFTTrainer) and args.bf16 is False and len(trainer.train_dataset) == N_TRAIN,
      "Trainer ready, T4-safe, on all your rows.",
      "Reuse the config and trainer cells from Lesson 5 (or Lesson 6 for LoRA). Keep bf16 off and train on `train`.")

In [ ]:
import time
t0 = time.time()
trainer.train()
print(f"trained in {time.time() - t0:.0f} s")
final = trainer.model.merge_and_unload() if isinstance(trainer.model, PeftModel) else trainer.model

## 4 · Evaluate against the bars

In [ ]:
print("FINE-TUNED model on the acceptance test:")
after, failures = score_v2(final, tok, test)
check(after["valid_json"] >= 0.95, "Bar 1 passed: valid-JSON rate ≥ 0.95.",
      "Bar 1 missed. Look at the replies below: is the format broken, or cut off?")
check(after["field_accuracy"] >= 0.90, "Bar 2 passed: field accuracy ≥ 0.90.",
      "Bar 2 missed. Tag the failures below (Lesson 8), fix your data in section 2, and rerun from there.")
print(f"\n{len(failures)} rows with at least one wrong field. The first 10:")
for f in failures[:10]:
    print(f"  text:     {f['text']}\n  reply:    {f['reply']}\n  expected: {f['expected']}\n")

**Probe it yourself.** Try messages unlike anything in either set. Note what breaks: it goes in your model card.

In [ ]:
def try_it(text):
    messages = [{"role": "user", "content": f"{PROMPT_V2}\n\nText: {text}"}]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(final.device)
    with torch.no_grad():
        out = final.generate(**inputs, max_new_tokens=60, do_sample=False)
    print(tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip())

try_it("Kwame, 34, just moved to Kumasi. Mail: kwame@example.com")

## 5 · Ship it (Lesson 9)
Save, reload and re-score what you'll ship, then write the card. Edit the card's text so it describes **your** data.

In [ ]:
import os, gc
from huggingface_hub import ModelCard, get_token, whoami

out_dir = "smollm2-json-extractor-v2"
final.save_pretrained(out_dir)
tok.save_pretrained(out_dir)
trainer = model = final = None; gc.collect(); torch.cuda.empty_cache()
shipped = AutoModelForCausalLM.from_pretrained(out_dir, dtype=torch.float32).to(device)
print("Reloaded from disk:")
shipped_scores, _ = score_v2(shipped, tok, test, show=0)
check(shipped_scores == after, "Same scores after saving and reloading: ship exactly this.",
      "The reloaded model scores differently. Rerun sections 3 to 5 in order.")

repo_id = None
if get_token():
    repo_id = f"{whoami()['name']}/{out_dir}"
    print("Will push to https://huggingface.co/" + repo_id)
else:
    print("No token: the push will skip. Lesson 9 shows how to add HF_TOKEN.")

**TODO 3:** Write your limitations, from section 4 and your probes. The cell then wraps `card_text` in a `ModelCard` and saves it as `README.md`.

In [ ]:
# TODO 3
limitations = ...

name = repo_id or f"your-username/{out_dir}"
card_text = f"""---
base_model: {model_id}
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
tags: [json, information-extraction, sft]
---

# {name}

Pulls `name`, `age`, `city` and `email` (or `null`) out of a short English message and replies with JSON only.
Fine-tuned from [{model_id}](https://huggingface.co/{model_id}).

## Training data
{N_TRAIN} synthetic messages from {len(templates)} templates, {len(TRAIN_NAMES)} names and {len(TRAIN_CITIES)} cities.

## Evaluation
60 held-out messages with unseen names, cities, email domains and phrasings. Greedy decoding.

| model | valid-JSON rate | field accuracy |
|---|---|---|
| base | {before['valid_json']:.2f} | {before['field_accuracy']:.2f} |
| this model | {after['valid_json']:.2f} | {after['field_accuracy']:.2f} |

## Limitations
{limitations}
"""

card = ModelCard(card_text)

card.save(os.path.join(out_dir, "README.md"))

In [ ]:
# check 3
check(isinstance(limitations, str) and len(limitations.split()) >= 8, "Limitations written.",
      "Write at least one full sentence about where your model breaks.")
check(isinstance(card, ModelCard) and card.data.base_model == model_id, "Card built with base_model metadata.",
      "The metadata didn't parse. Keep the --- block unindented at the very top of card_text.")

**TODO 4:** Push `shipped` and `tok` to `repo_id`. The card goes last.

In [ ]:
if repo_id:
    # TODO 4
    ...
    card.push_to_hub(repo_id)
    print("Live at https://huggingface.co/" + repo_id)
else:
    print("Skipped: no token.")

In [ ]:
# check 4
if repo_id:
    from huggingface_hub import HfApi
    api = HfApi()
    on_hub = set(api.list_repo_files(repo_id)) if api.repo_exists(repo_id) else set()
    check({"model.safetensors", "tokenizer.json", "README.md"} <= on_hub
          and ModelCard.load(repo_id).data.base_model == model_id,
          "Model, tokenizer and your card are on the Hub.",
          "Something is missing. Push the model and the tokenizer (TODO 4); the card goes last.")
else:
    check(False, "", "Nothing pushed: no token. Add HF_TOKEN (Lesson 9), then rerun section 5.")

## 6 · The finish line: anyone can load it in three lines

In [ ]:
from transformers import pipeline
extractor = pipeline("text-generation", model=repo_id or out_dir)
out = extractor([{"role": "user", "content": f"{PROMPT_V2}\n\nText: Zara Okafor, 29, lives in Kuala Lumpur."}], max_new_tokens=60)

print(out[0]["generated_text"][-1]["content"])

## Stretch (optional)
1. **Your own schema.** Add a field (say `phone`), write a matching held-out test generator with its own names, cities and phrasings, and hold your model to the same bars.
2. **Real data.** [`gretelai/gretel-pii-masking-en-v1`](https://huggingface.co/datasets/gretelai/gretel-pii-masking-en-v1) (Apache-2.0, synthetic documents with labelled PII) can be turned into prompt-completion rows. Its `entities` column is a Python-style string: parse it with `ast.literal_eval`, not `json.loads`.
3. **Bigger isn't free.** Compare the 135M and 360M recipes on the bars, training time and file size. Put the winner's numbers in the card.